## Example 1: assertExpectedInline passes when the value equals the inline literal

In [ ]:
import io
import os
import unittest

import expecttest

# Example 1: a TestCase whose expectations are string literals written in the test itself
os.environ.pop("EXPECTTEST_ACCEPT", None)


class Arithmetic(expecttest.TestCase):
    def test_sum(self):
        self.assertExpectedInline(str(sum(range(10))), """45""")

    def test_multi_line(self):
        self.assertExpectedInline("\n".join(["a", "b"]) + "\n", """\
a
b
""")


result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(Arithmetic))
assert result.wasSuccessful() and result.testsRun == 2, (result.failures, result.errors)
print(f"Example 1 OK: {result.testsRun} inline expectations held")

## Example 2: A mismatch is reported as a diff, with the hint to accept it

In [ ]:
import os

import expecttest

# Example 2: without EXPECTTEST_ACCEPT a wrong expectation raises AssertionError with a unified diff
os.environ.pop("EXPECTTEST_ACCEPT", None)
try:
    expecttest.assert_expected_inline("actual line\n", """expected line\n""")
except AssertionError as error:
    message = str(error)
else:
    raise AssertionError("a mismatch was not reported")
assert "-expected line" in message and "+actual line" in message, message
assert "EXPECTTEST_ACCEPT=1" in message, message
print("Example 2 OK: mismatch reported as a diff:", message.splitlines()[0])

## Example 3: EXPECTTEST_ACCEPT=1 rewrites the literal in the test file

In [ ]:
import os
import pathlib
import subprocess
import sys
import tempfile

# Example 3: run a test file with EXPECTTEST_ACCEPT=1; expecttest writes the actual value into its source
test_source = (
    "import unittest\nimport expecttest\n\n"
    "class T(expecttest.TestCase):\n"
    "    def test_upper(self):\n"
    "        self.assertExpectedInline('hello'.upper(), '''wrong''')\n\n"
    "unittest.main()\n"
)
with tempfile.TemporaryDirectory() as tmp_dir:
    path = pathlib.Path(tmp_dir) / "test_accept.py"
    path.write_text(test_source)
    env = {key: value for key, value in os.environ.items() if key != "EXPECTTEST_ACCEPT"}
    assert subprocess.run([sys.executable, str(path)], env=env, capture_output=True).returncode == 1
    accepted = subprocess.run([sys.executable, str(path)], env={**env, "EXPECTTEST_ACCEPT": "1"}, capture_output=True)
    assert accepted.returncode == 0, accepted.stderr.decode()
    rewritten = path.read_text()
    assert "'''HELLO'''" in rewritten and "'''wrong'''" not in rewritten, rewritten
    assert subprocess.run([sys.executable, str(path)], env=env, capture_output=True).returncode == 0
print("Example 3 OK: the literal was rewritten to '''HELLO''' and the test now passes")

## Example 4: assertExpectedRaisesInline checks an exception's message

In [ ]:
import io
import os
import unittest

import expecttest

# Example 4: the raised exception's str() is compared with the literal; no exception is a failure
os.environ.pop("EXPECTTEST_ACCEPT", None)


class Parsing(expecttest.TestCase):
    def test_bad_int(self):
        self.assertExpectedRaisesInline(ValueError, lambda: int("x"), """invalid literal for int() with base 10: 'x'""")

    def test_no_exception(self):
        self.assertExpectedRaisesInline(ValueError, lambda: int("7"), """never raised""")


result = unittest.TextTestRunner(stream=io.StringIO(), verbosity=0).run(
    unittest.defaultTestLoader.loadTestsFromTestCase(Parsing))
assert result.testsRun == 2 and not result.errors, result.errors
assert len(result.failures) == 1 and "test_no_exception" in str(result.failures[0][0]), result.failures
assert "Did not raise when expected to" in result.failures[0][1]
print("Example 4 OK: message checked; a call that does not raise fails the test")

## Example 5: The building blocks: Expect objects and string-literal rewriting

In [ ]:
import os

import expecttest

# Example 5: an Expect value can be passed around (parametrised tests); replace_string_literal is the rewriter
os.environ.pop("EXPECTTEST_ACCEPT", None)
expectation = expecttest.Expect("42")
expectation.assert_expected(str(6 * 7))
assert str(expectation) == "42" and repr(expectation) == "Expect('42')"

new_source, delta = expecttest.replace_string_literal("x = '''old'''", 1, 1, "line one\nline two\n")
namespace = {}
exec(new_source, namespace)
assert namespace["x"] == "line one\nline two\n", (new_source, namespace["x"])
assert delta == new_source.count("\n"), (delta, new_source)
print(f"Example 5 OK: Expect held; the literal became {len(new_source.splitlines())} lines (delta {delta})")